# ¡PELIGRO! Documento en obras

In [8]:
from threading import Thread, Event, Lock
import time

In [9]:
c = "*"   # Variable global a la que queremos acceder en mutex
l = Lock() # Cerrojo para el acceso mutex a c
e = Event() # Evento para anunciar el paso de 30 segundos
salir = Event()  # Evento para indicar que se debe salir del programa

In [10]:
# Cada x segundos imprime el número de segundo por el que va y alterna c entre * y +. Termina cuando llega a 60 segundos
def funchebra1(x):
    global c
    global e
    print("Incio de hebra 1")
    
    inicio = time.time()
    contador = 0
    while (time.time()-inicio)<60:
        if contador == 30:
            e.set()
        if contador % x == 0:
            print(contador)
            l.acquire()
            if c == "*":
                c = "+"
            else:
                c = "*"
            l.release()
        while (time.time()-inicio) < contador + 1:
            time.sleep(0.1)
        contador += 1
    print("Fin de hebra 1")

In [11]:
# Cada segundo imprime el valor de c. Termina cuando se activa el evento salir
def funchebra2():
    global salir
    global c
    print("Incio de hebra 2")
    while not salir.is_set():
        time.sleep(1)
        l.acquire()
        print(c)
        l.release()
    print("Fin de hebra 2")

In [12]:
# Espera a que se produzca el evento e para imprimir una línea
def funchebra3():
    global e
    print("Incio de hebra 3")
    e.wait()
    print("--------------------------------------------")
    e.clear()
    salir.set()
    print("Fin de hebra 3")

In [13]:
hebra1 = Thread(target=funchebra1, args=[5])
hebra2 = Thread(target=funchebra2)
hebra3 = Thread(target=funchebra3)

In [14]:
hebra1.start()
hebra2.start()
hebra3.start()

# Esperamos a que se produzca el evento salir para terminar el programa
while not salir.is_set():    
    pass

print("Esperando a que acabe la hebra 1")
hebra1.join()
print("Final del programa")

Incio de hebra 1
0
Incio de hebra 2
Incio de hebra 3
+
+
+
+
+
5
*
*
*
*
10
+
+
+
+
+
15
*
*
*
*
*
20
+
+
+
+
+
25
*
*
*
*
*
30
--------------------------------------------
Fin de hebra 3
Esperando a que acabe la hebra 1
+
Fin de hebra 2
35
40
45
50
55
Fin de hebra 1
Final del programa
